download dataset from kaggle


In [ ]:
!curl -L -o dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/shaunthesheep/microsoft-catsvsdogs-dataset

!unzip dataset.zip

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os
from tensorflow.keras.preprocessing import image
import numpy as np

check for corrupted files


In [11]:
from PIL import Image

def check_images_in_folder(folder_path):
    corrupted_images = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                try:
                    img_path = os.path.join(root, file)
                    img = Image.open(img_path)
                    img.verify()  # Verify that the image is not corrupted
                    img.close()
                except Exception as e:
                    print(f"Corrupted image: {img_path} - {e}")
                    corrupted_images.append(img_path)
    return corrupted_images

DATASET_PATH = "PetImages"  # Path to your dataset
corrupted_images = check_images_in_folder(DATASET_PATH)
print(f"Found {len(corrupted_images)} corrupted images.")

/opt/anaconda3/envs/ml-env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Found 0 corrupted images.


In [10]:
!rm PetImages/Cat/666.jpg PetImages/Dog/11702.jpg

**Training**

In [ ]:
# Define paths
DATASET_PATH = "PetImages"  # Root folder containing "Cat" and "Dog"
BATCH_SIZE = 16  # Reduced batch size
IMG_SIZE = (128, 128)  # Reduced image size

# Data Preprocessing
datagen = ImageDataGenerator(
    rescale=1.0/255,  # Normalize pixel values
    validation_split=0.2  # 80% training, 20% validation
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

# Load Pretrained Model (MobileNetV2)
base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
base_model.trainable = False  # Freeze the base model

# Create Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dense(1, activation="sigmoid")(x)  # Binary classification

# Build the Model
model = Model(inputs=base_model.input, outputs=x)

# Compile the Model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the Model
EPOCHS = 3  # Reduced number of epochs
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

# Save the Model
model.save("dog_cat_classifier.keras")

print("Training complete. Model saved as dog_cat_classifier.keras")

Found 20000 images belonging to 2 classes.
Found 4998 images belonging to 2 classes.
Epoch 1/3


/opt/anaconda3/envs/ml-env/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 60s 46ms/step - accuracy: 0.9472 - loss: 0.1583 - val_accuracy: 0.9578 - val_loss: 0.1077
Epoch 2/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 56s 45ms/step - accuracy: 0.9688 - loss: 0.0821 - val_accuracy: 0.9636 - val_loss: 0.1002
Epoch 3/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 55s 44ms/step - accuracy: 0.9718 - loss: 0.0763 - val_accuracy: 0.9608 - val_loss: 0.1080
Training complete. Model saved as dog_cat_classifier.keras


**Inference**

In [10]:
def predict_image(img_path, model):
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]
    print(f"Prediction 1 - Dog, 0 - Cat: {prediction}")
    return "Dog" if prediction > 0.5 else "Cat"

model = tf.keras.models.load_model("dog_cat_classifier.keras")
print(predict_image("test data/cat2.jpeg", model))  # Replace with actual image path


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction 1 - Dog, 0 - Cat: 0.00016902964853215963
Cat
